# Basketball Noodling

Computing plus-minus using play-by-play data. 

In [1]:
import pandas as pd

from bs4 import BeautifulSoup
import re
from pprint import pprint

import requests
from pathlib import Path

## Get a few box scores

In [2]:
data_path = Path("/Users/rory/data/ncaa_stats")

cu_game_ids = [
    5736833,
    5731718,
    5736837,
]

In [ ]:
def box_score_html(game_id, field):
    """Downloads and saves a box score HTML page from NCAA Stats

    Parameters
    ----------
    game_id : int
        NCAA game ID
    field : str
        Either play_by_play or box_score

    Returns
    -------
    str
        Content of the downloaded HTML file, or None if an error occurred.
    """
    url_path = f"contests/{game_id}/{field}"
    html_path = data_path / f"{url_path}.html"
    if html_path.exists():
        print("Got this one", game_id)
        return html_path.read_text()

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    url = f"https://stats.ncaa.org/{url_path}"
    r = requests.get(url, headers=headers)

    if not r.ok:
        print("Problem", r.text)

    html_path.parent.mkdir(parents=True, exist_ok=True)
    html_path.write_bytes(r.content)
    return r.text

In [16]:
html_content = box_score_html(5736837, "play_by_play")

Got this one 5736837


In [17]:
soup = BeautifulSoup(html_content)

In [18]:
tables = soup.find_all("table", class_="table")

all_plays = []

for table in tables:
    # Get period name from previous card-header
    period = table.find_previous("div", class_="card-header").get_text(strip=True)

    for tr in table.tbody.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) == 4:
            time = tds[0].get_text(strip=True)
            team1_play = tds[1].get_text(strip=True)
            score = tds[2].get_text(strip=True)
            team2_play = tds[3].get_text(strip=True)

            # Skip empty rows
            if not any([team1_play, team2_play]):
                continue

            all_plays.append(
                {
                    "Period": period,
                    "Time": time,
                    "Team1_Play": team1_play,
                    "Score": score,
                    "Team2_Play": team2_play,
                }
            )

# Convert to DataFrame
df = pd.DataFrame(all_plays)

In [67]:
df[df.Team2_Play.str.contains("sub")].head(10).Team2_Play.to_list()

['Anaëlle Dutat,substitution out',
 'Sophia Zadel,substitution out',
 'Desiree Wooten,substitution in',
 'Logyn Greer,substitution in',
 'Kennedy Sanders,substitution out',
 'Maeve McErlane,substitution in',
 'Jade Crook,substitution out',
 'Kennedy Sanders,substitution in',
 'Anaëlle Dutat,substitution in',
 'Zyanna Walker,substitution out']

In [73]:
# team = set()


def update_team(team, play):
    [player, direction] = play.split(",")
    is_out = "out" in direction

    if is_out:
        try:
            team.remove(player)
        except KeyError:
            if "Unknown 1" in team:
                team.add("Unknown 2")
            else:
                team.add("Unknown 1")
    else:
        team.add(player)

    if len(team) > 5:
        try:
            team.remove("Unknown 2")
        except KeyError:
            team.remove("Unknown 1")

    return team


# update_team(team, "Anaëlle Dutat,substitution out")
update_team(team, "Desiree Wooten,substitution in")

{'Desiree Wooten',
 'Kennedy Sanders',
 'Logyn Greer',
 'Maeve McErlane',
 'Unknown 1'}

In [74]:
team = set()
plays = [
    "Anaëlle Dutat,substitution out",
    "Sophia Zadel,substitution out",
    "Desiree Wooten,substitution in",
    "Logyn Greer,substitution in",
    "Kennedy Sanders,substitution out",
    "Maeve McErlane,substitution in",
    "Jade Crook,substitution out",
    "Kennedy Sanders,substitution in",
    "Anaëlle Dutat,substitution in",
    "Zyanna Walker,substitution out",
]

for p in plays:
    team = update_team(team, p)
    print(team)

{'Unknown 1'}
{'Unknown 1', 'Unknown 2'}
{'Unknown 1', 'Unknown 2', 'Desiree Wooten'}
{'Unknown 1', 'Unknown 2', 'Desiree Wooten', 'Logyn Greer'}
{'Unknown 1', 'Unknown 2', 'Desiree Wooten', 'Logyn Greer'}
{'Unknown 1', 'Unknown 2', 'Maeve McErlane', 'Logyn Greer', 'Desiree Wooten'}
{'Unknown 1', 'Unknown 2', 'Maeve McErlane', 'Logyn Greer', 'Desiree Wooten'}
{'Unknown 1', 'Maeve McErlane', 'Kennedy Sanders', 'Logyn Greer', 'Desiree Wooten'}
{'Maeve McErlane', 'Kennedy Sanders', 'Anaëlle Dutat', 'Logyn Greer', 'Desiree Wooten'}
{'Maeve McErlane', 'Kennedy Sanders', 'Anaëlle Dutat', 'Logyn Greer', 'Desiree Wooten'}


In [46]:
team.remove("hi")

KeyError: 'hi'

In [ ]:
set().remove()

In [29]:
html_content = box_score_html(5736837, "box_score")
soup = BeautifulSoup(html_content)

# Updated regex pattern for the specific `addShot` format you provided
shot_data_pattern = re.compile(
    r"addShot\(\s*([\d.]+),\s*([\d.]+),\s*(\d+),\s*(true|false),\s*(\d+),\s*'([^']+)',\s*'([^']+)',\s*(true|false)\s*\);"
)

shot_data = []

for script in soup.find_all("script"):
    if script.string and shot_data_pattern.search(script.string):
        for match in shot_data_pattern.findall(script.string):
            # Map matched groups to relevant fields
            shot = {
                "x": float(match[0]),  # X-coordinate
                "y": float(match[1]),  # Y-coordinate
                "team_id": int(match[2]),  # Team ID
                "made": match[3] == "true",  # Shot success (True if 'made')
                "player_id": int(match[4]),  # Player ID
                "description": match[5],  # Shot description
                "meta_info": match[6],  # Additional metadata
                "flag": match[7] == "true",  # Extra boolean flag, if needed
            }
            shot_data.append(shot)


# Extract teams and players from the dropdowns
teams = {}
for option in soup.select("#team_select option"):
    if option.get("value").isdigit():
        teams[int(option["value"])] = option.text.strip()

players = {}
for optgroup in soup.select("#player_select optgroup"):
    team = optgroup.get("label")
    players[team] = {
        int(option["value"]): option.text.strip()
        for option in optgroup.find_all("option")
    }

# Print results
# print("Shot Data:", shot_data)
print("Teams:", teams)
print("Players by Team:")
pprint(players)

Got this one 5736837
Teams: {157: 'Colorado', 66: 'Boise St.'}
Players by Team:
{'Boise St.': {776307800: 'Mary Kay Naro',
               776307801: 'Elodie Lalotte',
               776307802: 'Abby Muse',
               776307803: 'Jayda Clark',
               776307804: 'Tatum Thompson',
               776307805: 'Mya Hansen',
               776307806: 'Dani Bayes',
               776307807: 'Natalie Pasco',
               776307808: 'Alyssa Christensen',
               776307809: 'Libby Hutton',
               776307810: 'Teryn Gardner'},
 'Colorado': {776307813: 'Nyamer Diew',
              776307814: 'Lior Garzon',
              776307816: 'Johanna Teder',
              776307818: 'Sara-Rose Smith',
              776307820: 'Kindyll Wetta',
              776307822: 'Jade Masogayo',
              776307824: 'Kennedy Sanders',
              776307827: 'Ayianna Johnson',
              776307828: 'Tabitha Betson'}}


In [26]:
shot_df = pd.DataFrame(shot_data)
shot_df.head()

""


In [28]:
html_content

'<!DOCTYPE html><html><head> <meta charset="utf-8"> <meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no"> <meta http-equiv="refresh" content="5; URL=\'/contests/5736837/box_score?bm-verify=AAQAAAAM_____2X1MEz_n4SrVbDmzJ7gVKm8NTAd0UFmql8ABToMUm0C1VbZqise6TZPGALup2x2c6mr7TPHhIJz6C6JaPcN7ROYXDkjlMKL4wa8F0uFAgu6_36PjGRpjLzQftb1nyK-x7YPLGm5dfwhlDkTewX-Zk81fbuDJ2RUDbXI-H5mbH0NvjbIySBKM_dd1X4vGu-AEAaoTDdYPEYrS9fKgCpfS95wtNPfGyk2SDM2V2sX8Vciab1JgZilhhIzwJJTpkP1YaZm0fy7GAfO\'" /><title>&nbsp;</title><script> var i = 1767640577; var j = i + Number("7187" + "18349"); </script> </head> <noscript> <iframe style="border: none; height: 100%; width: 100%;" src="https://stats.ncaa.org/request_quota_reached.html"></iframe></noscript><body> <iframe style="border: none; width: 100vw; height: 100vh;" src="https://stats.ncaa.org/akamai_validation.html"> </iframe> <script> function triggerInterstitialChallenge() {var xhr = new XMLHttpRequest(); xhr.withCredentials = true; xhr